In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from scipy.stats import linregress
import matplotlib
import scipy.optimize as opt

# Метод Бесселя

In [22]:
L1 = 869
d_L1 = 2
l1_1 = 389
d_l1_1 = 2
l1_2 = 388
d_l1_2 = 2

L2 = 1115
d_L2 = 3
l2_1 = 682
d_l2_1 = 3
l2_2 = 685
d_l2_2 = 3

d = 13
n = 1.5
delta = 3.5


In [23]:
def f(L, dL, l, dl):
    f = (L**2 - l**2)/4/L
    df = (((2 * L * dL)**2 + (2 * l * dl)**2)/(L**2 - l**2)**2 + dL**2/L**2)**0.5 * f
    return (f, df)
def sigma_f(L, l, delta):
    return (L**2 + l**2)/(L**2 - l**2) * delta/L

f(L2, d_L2, l2_2, d_l2_2), sigma_f(L2, l2_2, delta)

((173.542600896861, 1.8213270010710656), 0.006944965875250578)

In [29]:
sigma_f(L2, l2_1, delta)

0.006891818020120193

# Телескоп Галилея

In [30]:
def alpha(x1, y1, x2, y2, X1, Y1, X2, Y2, n, dx1 = 0, dy1 = 0, dx2 = 0, dy2 = 0, dX1 = 0, dY1 = 0, dX2 = 0, dY2 = 0):
    R = ((X1 - X2)**2 + (Y1 - Y2)**2)**0.5
    dR = (abs(X1 - X2)*(dX1 + dX2) + abs(Y1 - Y2)*(dY1 + dY2))/R
    r = ((x1 - x2)**2 + (y1 - y2)**2)**0.5
    dr = (abs(x1 - x2)*(dx1 + dx2) + abs(y1 - y2)*(dy1 + dy2))/r
    return (R/n/r, ((dR/R)**2 + (dr/r)**2)**0.5 * R/n/r)
# после коллиматора
(x1, y1) = (332, 206)
(x2, y2) = (326, 509)
(X1, Y1) = (322, 189)
(X2, Y2) = (306, 530)
n = 14
alpha0, d_alpha0 = alpha(x1, y1, x2, y2, X1, X2, Y1, Y2, n)
# после телескопа
(x1, y1) = (341, 205)
(x2, y2) = (341, 503)
(X1, Y1) = (305, 121)
(X2, Y2) = (298, 554)
n = 10
alpha1, d_alpha1 = alpha(x1, y1, x2, y2, X1, X2, Y1, Y2, n, dY1 = 2, dY2 = 2)

#увеличение напрямую
gamma_pic = alpha1/alpha0
d_gamma_pic = d_alpha1/alpha0

In [33]:
f_ob = 174
d_f_ob = 1
f_ok = 96
d_f_ok = 4

#увеличение по фокусам
gamma_f = f_ob/f_ok
d_gamma_f = ((d_f_ob/f_ob)**2 + (d_f_ok/f_ok)**2)**0.5 * gamma_f

In [34]:
D0 = 45
d_D0 = 1
D1 = 21
d_D1 = 1

#увеличение по диаметрам
gamma_D = D0/D1
d_gamma_D = ((d_D0/D0)**2 + (d_D1/D1)**2)**0.5*gamma_D

In [35]:
print("реальное увеличение", gamma_pic, d_gamma_pic)
print("увеличение по фокусам", gamma_f, d_gamma_f)
print("увеличение по диаметрам", gamma_D, d_gamma_D)

реальное увеличение 1.7230279365147032 0.017751736628612524
увеличение по фокусам 1.8125 0.07623583941825232
увеличение по диаметрам 2.142857142857143 0.11260507045746154


# Микроскоп

In [38]:
L_zr = 25
f_k= 25.0
d_f_k = 0.2
f_ob = 7.9
d_f_ob = 0.2
f_ok = 5.4
d_f_ok = 0.2

# после микроскопа
(x1, y1) = (340, 205)
(x2, y2) = (340, 512)
(X1, Y1) = (223, 107)
(X2, Y2) = (218, 638)
n = 2
alpha2, d_alpha2 = alpha(x1, y1, x2, y2, X1, X2, Y1, Y2, n, dX1 = 7, dY1 = 9, dX2 = 6, dY2 = 8)
gamma_t = alpha2/alpha0*L_zr/f_k
d_gamma_t = ((d_alpha2/alpha2)**2 + (d_f_k/f_k)**2)**0.5 * gamma_t

#теоретическое увеличение
L = 28.9
d_L = 0.2
Delta = L - f_ok - f_ob
d_Delta = (d_L**2 + d_f_ok**2 + d_f_ob**2)**0.5
gamma_f = Delta/f_ok * L_zr/f_ob
d_gamma_f = ((d_Delta/Delta)**2 + (d_f_ok/f_ok)**2 + (d_f_ob/f_ob)**2)**0.5 * gamma_f

print("оптический интервал", Delta, d_Delta)
print("измеренное увеличение", gamma_t, d_gamma_t)
print("теоретическое увеличение", gamma_f, d_gamma_f)

оптический интервал 15.6 0.3464101615137755
измеренное увеличение 11.557839530139768 0.5345211399626943
теоретическое увеличение 9.142053445850912 0.45762910578940813
